# Step 2 — State-Level Competitiveness
**Objective:** Disaggregate Brazil's national trade picture to the state level — identifying which states drive exports and imports, which sectors and products dominate each region, how trade balances vary, and whether the China dependency identified in Step 1 is uniform or concentrated.

This notebook covers:
1. Total exports and imports by state (2025)
2. Top exporting states over time
3. Trade balance by state (2025)
4. State trade balance evolution across key periods
5. Export distribution by Brazilian region
6. Top sectors by state — exports (SH2 classification)
7. Top sectors by state — imports (SH2 classification)
8. Top NCM products by state and region
9. State-level China dependency
10. Monthly trade variation by state — exports (box plot)
11. Monthly trade variation by state — imports (box plot)
12. State trade hubs — export and import profiles
13. Regional trade hubs — export and import profiles
14. Commodities vs processed products by state and region [PLACEHOLDER — requires classification table]
15. Key Findings

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Credentials
dotenv_path = Path(r"C:\Users\e_koh\Downloads\State Analysis\brazil-state-trade-analysis\.env")
load_dotenv(dotenv_path, override=True)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Connected to database successfully")

## Data capped at 2025 — 2026 contains only partial year data which distorts trend visuals
MAX_YEAR  = 2025
MIN_YEAR  = 1997
## Monthly box plot window — last 10 years
BOX_START = 2016

## 2.1 — Total Exports and Imports by State (2025)
Which states are Brazil's largest traders in the most recent full year?

In [ ]:
query_state_trade = f"""
    SELECT u.nome_estado AS state,
           u.nome_regiao AS region,
           u.sigla AS uf,
           COALESCE(e.exports_usd, 0) AS exports_usd,
           COALESCE(i.imports_usd, 0) AS imports_usd,
           COALESCE(e.exports_usd, 0) - COALESCE(i.imports_usd, 0) AS balance_usd
    FROM uf u
    LEFT JOIN (
        SELECT "SG_UF_NCM", SUM("VL_FOB") AS exports_usd
        FROM exp
        WHERE "CO_ANO" = {MAX_YEAR}
        GROUP BY "SG_UF_NCM"
    ) e ON e."SG_UF_NCM" = u.sigla
    LEFT JOIN (
        SELECT "SG_UF_NCM", SUM("VL_FOB") AS imports_usd
        FROM imp
        WHERE "CO_ANO" = {MAX_YEAR}
        GROUP BY "SG_UF_NCM"
    ) i ON i."SG_UF_NCM" = u.sigla
    WHERE COALESCE(e.exports_usd, 0) > 0 OR COALESCE(i.imports_usd, 0) > 0
    ORDER BY exports_usd DESC
"""

df_state = pd.read_sql(query_state_trade, engine)
df_state['exports_usd_bn'] = (df_state['exports_usd'] / 1e9).round(2)
df_state['imports_usd_bn'] = (df_state['imports_usd'] / 1e9).round(2)
df_state['balance_usd_bn'] = (df_state['balance_usd'] / 1e9).round(2)

df_top15 = df_state.head(15).sort_values('exports_usd_bn')

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(df_top15['state'], df_top15['exports_usd_bn'], color='steelblue', alpha=0.85, label='Exports')
ax.barh(df_top15['state'], df_top15['imports_usd_bn'], color='tomato', alpha=0.5, label='Imports')
ax.set_title(f"Brazil Top 15 States by Export Value ({MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_xlabel("Trade Value (USD Billions)")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))
ax.legend()
plt.tight_layout()
plt.savefig('output_2_1_state_trade.png', dpi=150)
plt.show()

print(f"\nAll states — Exports, Imports and Balance ({MAX_YEAR})")
print(df_state[['state', 'region', 'uf', 'exports_usd_bn', 'imports_usd_bn', 'balance_usd_bn']].to_string(index=False))

## 2.2 — Top Exporting States Over Time
How has the ranking of Brazil's top exporting states shifted from 1997 to 2025?

In [ ]:
query_state_time = f"""
    SELECT e."CO_ANO" AS year,
           u.nome_estado AS state,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    WHERE e."CO_ANO" <= {MAX_YEAR}
    GROUP BY e."CO_ANO", u.nome_estado
    ORDER BY e."CO_ANO", exports_usd DESC
"""

df_state_time = pd.read_sql(query_state_time, engine)
df_state_time['exports_usd_bn'] = df_state_time['exports_usd'] / 1e9

top8_states = (
    df_state_time.groupby('state')['exports_usd']
    .sum().nlargest(8).index.tolist()
)

df_top8 = df_state_time[df_state_time['state'].isin(top8_states)]
df_pivot = df_top8.pivot(index='year', columns='state', values='exports_usd_bn').fillna(0)

## Annotate key periods identified in Step 1
events = {
    2009: ('2009\nTrade Contraction', 2008.5),
    2011: ('2011\nExport Peak',       2011.5),
    2014: ('2014\nRecession',         2014.0),
    2020: ('COVID-19',                2019.833)
}

fig, ax = plt.subplots(figsize=(14, 7))
for state in df_pivot.columns:
    ax.plot(df_pivot.index, df_pivot[state], label=state, linewidth=2)
for year, (label, xpos) in events.items():
    ax.axvline(x=xpos, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(xpos + 0.1, ax.get_ylim()[1] * 0.85, label, fontsize=7, color='gray')

ax.set_title(f"Brazil Top 8 Exporting States (1997–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Export Value (USD Billions)")
ax.legend(loc='upper left', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))
plt.tight_layout()
plt.savefig('output_2_2_state_exports_time.png', dpi=150)
plt.show()

## 2.3 — Trade Balance by State (2025)
Which states are net exporters and which are net importers?

In [ ]:
## Uses df_state from 2.1
df_balance = df_state[['state', 'uf', 'region', 'balance_usd_bn']].sort_values('balance_usd_bn')
colors = ['tomato' if x < 0 else 'steelblue' for x in df_balance['balance_usd_bn']]

fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(df_balance['state'], df_balance['balance_usd_bn'], color=colors, alpha=0.85)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title(f"Trade Balance by State ({MAX_YEAR}) — Surplus vs Deficit", fontsize=14, fontweight='bold')
ax.set_xlabel("Trade Balance (USD Billions)")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))
plt.tight_layout()
plt.savefig('output_2_3_state_balance.png', dpi=150)
plt.show()

surplus_states = df_balance[df_balance['balance_usd_bn'] > 0].sort_values('balance_usd_bn', ascending=False)
deficit_states = df_balance[df_balance['balance_usd_bn'] < 0].sort_values('balance_usd_bn')

print(f"Net exporting states ({MAX_YEAR}):")
print(surplus_states[['state', 'region', 'balance_usd_bn']].to_string(index=False))
print(f"\nNet importing states ({MAX_YEAR}):")
print(deficit_states[['state', 'region', 'balance_usd_bn']].to_string(index=False))

## 2.4 — State Trade Balance Evolution Across Key Periods
Step 1 showed Brazil entered a national trade deficit around 2013. Which states drove that shift? How did state-level balances change across the key periods identified in Step 1?

In [ ]:
## Key periods from Step 1 analysis
key_years = [1997, 2011, 2014, 2020, MAX_YEAR]

query_balance_periods = f"""
    SELECT year, uf, state,
           COALESCE(exports_usd, 0) - COALESCE(imports_usd, 0) AS balance_usd
    FROM (
        SELECT e."CO_ANO" AS year,
               u.sigla AS uf,
               u.nome_estado AS state,
               SUM(e."VL_FOB") AS exports_usd,
               NULL::numeric AS imports_usd
        FROM exp e
        JOIN uf u ON e."SG_UF_NCM" = u.sigla
        WHERE e."CO_ANO" IN ({','.join(map(str, key_years))})
        GROUP BY e."CO_ANO", u.sigla, u.nome_estado
        UNION ALL
        SELECT i."CO_ANO" AS year,
               u.sigla AS uf,
               u.nome_estado AS state,
               NULL::numeric AS exports_usd,
               SUM(i."VL_FOB") AS imports_usd
        FROM imp i
        JOIN uf u ON i."SG_UF_NCM" = u.sigla
        WHERE i."CO_ANO" IN ({','.join(map(str, key_years))})
        GROUP BY i."CO_ANO", u.sigla, u.nome_estado
    ) t
    GROUP BY year, uf, state
"""

df_periods = pd.read_sql(query_balance_periods, engine)
df_periods['balance_usd_bn'] = (df_periods['balance_usd'] / 1e9).round(2)

## Pivot: rows = states, columns = years
df_periods_pivot = df_periods.pivot(index='state', columns='year', values='balance_usd_bn').fillna(0)

## Focus on top 15 states by absolute balance in MAX_YEAR
top15_balance = df_state.nlargest(15, 'exports_usd_bn')['state'].tolist()
df_periods_plot = df_periods_pivot.loc[df_periods_pivot.index.isin(top15_balance)]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(key_years))
width = 0.05
for i, state in enumerate(df_periods_plot.index):
    ax.plot(key_years, df_periods_plot.loc[state], marker='o', label=state, linewidth=1.5)

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title("State Trade Balance Across Key Periods", fontsize=14, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Trade Balance (USD Billions)")
ax.set_xticks(key_years)
ax.legend(loc='upper left', fontsize=7, ncol=2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))
plt.tight_layout()
plt.savefig('output_2_4_state_balance_periods.png', dpi=150)
plt.show()

print("\nState trade balance across key periods (USD bn):")
print(df_periods_plot.to_string())

## 2.5 — Export Distribution by Brazilian Region
How are exports distributed across Brazil's five macro-regions? How has this shifted from 1997 to 2025?

In [ ]:
query_region = f"""
    SELECT e."CO_ANO" AS year,
           u.nome_regiao AS region,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    WHERE e."CO_ANO" <= {MAX_YEAR}
    GROUP BY e."CO_ANO", u.nome_regiao
    ORDER BY e."CO_ANO"
"""

df_region = pd.read_sql(query_region, engine)
df_region['exports_usd_bn'] = df_region['exports_usd'] / 1e9
df_region_pivot = df_region.pivot(index='year', columns='region', values='exports_usd_bn').fillna(0)
df_region_pct   = df_region_pivot.div(df_region_pivot.sum(axis=1), axis=0) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for region in df_region_pivot.columns:
    ax1.plot(df_region_pivot.index, df_region_pivot[region], label=region, linewidth=2)
ax1.set_title(f"Exports by Region — Absolute Value (1997–{MAX_YEAR})", fontsize=12, fontweight='bold')
ax1.set_xlabel("Year")
ax1.set_ylabel("Export Value (USD Billions)")
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))

ax2.stackplot(df_region_pct.index, df_region_pct.T, labels=df_region_pct.columns, alpha=0.8)
ax2.set_title(f"Exports by Region — Share of Total (1997–{MAX_YEAR})", fontsize=12, fontweight='bold')
ax2.set_xlabel("Year")
ax2.set_ylabel("Share of Total Exports (%)")
ax2.legend(loc='upper left', fontsize=8)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
plt.tight_layout()
plt.savefig('output_2_5_region_exports.png', dpi=150)
plt.show()

start_year = df_region_pct.index[0]
summary_region = pd.DataFrame({
    'Region'            : df_region_pct.columns,
    f'{start_year} (%)' : df_region_pct.loc[start_year].round(1).values,
    f'{MAX_YEAR} (%)'   : df_region_pct.loc[MAX_YEAR].round(1).values,
    'Change (pp)'       : (df_region_pct.loc[MAX_YEAR] - df_region_pct.loc[start_year]).round(1).values
})
print(f"\nRegional export share — {start_year} vs {MAX_YEAR}")
print(summary_region.sort_values(f'{MAX_YEAR} (%)', ascending=False).to_string(index=False))

## 2.6 — Top Export Sectors by State (SH2 Classification)
What are the dominant export sectors for Brazil's top 10 exporting states?

In [ ]:
top10_states_uf = df_state.head(10)['uf'].tolist()
top10_states_uf_str = ','.join([f"'{s}'" for s in top10_states_uf])

query_exp_sectors = f"""
    SELECT e."SG_UF_NCM" AS uf,
           u.nome_estado AS state,
           s.codigo_sh2,
           s.descricao_sh2,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    JOIN ncm n ON e."CO_NCM" = n.codigo_ncm
    JOIN ncm_sh s ON n.codigo_sh6 = s.codigo_sh6
    WHERE e."CO_ANO" = {MAX_YEAR}
    AND e."SG_UF_NCM" IN ({top10_states_uf_str})
    GROUP BY e."SG_UF_NCM", u.nome_estado, s.codigo_sh2, s.descricao_sh2
    ORDER BY e."SG_UF_NCM", exports_usd DESC
"""

df_exp_sectors = pd.read_sql(query_exp_sectors, engine)
df_exp_sectors['exports_usd_bn'] = (df_exp_sectors['exports_usd'] / 1e9).round(2)

df_top10_sectors = (
    df_exp_sectors.groupby('state', group_keys=False)
    .apply(lambda x: x.nlargest(10, 'exports_usd'))
    .reset_index(drop=True)
)

print(f"Top 10 export sectors per state — SH2 classification ({MAX_YEAR})")
for state in df_top10_sectors['state'].unique():
    subset = df_top10_sectors[df_top10_sectors['state'] == state]
    total  = df_state[df_state['uf'] == subset['uf'].iloc[0]]['exports_usd_bn'].iloc[0]
    print(f"\n{state} — Total exports: ${total:.1f}bn")
    for _, row in subset.iterrows():
        share = (row['exports_usd'] / (total * 1e9) * 100)
        print(f"  SH{row['codigo_sh2']} {str(row['descricao_sh2'])[:50]:<50} ${row['exports_usd_bn']:.2f}bn ({share:.1f}%)")

## 2.7 — Top Import Sectors by State (SH2 Classification)
What are the dominant import sectors for Brazil's top 10 importing states?

In [ ]:
top10_imp_states_uf = df_state.nlargest(10, 'imports_usd_bn')['uf'].tolist()
top10_imp_states_uf_str = ','.join([f"'{s}'" for s in top10_imp_states_uf])

query_imp_sectors = f"""
    SELECT i."SG_UF_NCM" AS uf,
           u.nome_estado AS state,
           s.codigo_sh2,
           s.descricao_sh2,
           SUM(i."VL_FOB") AS imports_usd
    FROM imp i
    JOIN uf u ON i."SG_UF_NCM" = u.sigla
    JOIN ncm n ON i."CO_NCM" = n.codigo_ncm
    JOIN ncm_sh s ON n.codigo_sh6 = s.codigo_sh6
    WHERE i."CO_ANO" = {MAX_YEAR}
    AND i."SG_UF_NCM" IN ({top10_imp_states_uf_str})
    GROUP BY i."SG_UF_NCM", u.nome_estado, s.codigo_sh2, s.descricao_sh2
    ORDER BY i."SG_UF_NCM", imports_usd DESC
"""

df_imp_sectors = pd.read_sql(query_imp_sectors, engine)
df_imp_sectors['imports_usd_bn'] = (df_imp_sectors['imports_usd'] / 1e9).round(2)

df_top10_imp_sectors = (
    df_imp_sectors.groupby('state', group_keys=False)
    .apply(lambda x: x.nlargest(10, 'imports_usd'))
    .reset_index(drop=True)
)

print(f"Top 10 import sectors per state — SH2 classification ({MAX_YEAR})")
for state in df_top10_imp_sectors['state'].unique():
    subset = df_top10_imp_sectors[df_top10_imp_sectors['state'] == state]
    total  = df_state[df_state['uf'] == subset['uf'].iloc[0]]['imports_usd_bn'].iloc[0]
    print(f"\n{state} — Total imports: ${total:.1f}bn")
    for _, row in subset.iterrows():
        share = (row['imports_usd'] / (total * 1e9) * 100)
        print(f"  SH{row['codigo_sh2']} {str(row['descricao_sh2'])[:50]:<50} ${row['imports_usd_bn']:.2f}bn ({share:.1f}%)")

## 2.8 — Top NCM Products by State and Region
Beyond sector-level SH2 classification — what are the specific top 10 NCM products for each of the top 10 exporting states and each region?

In [ ]:
## ---- TOP NCM BY STATE ----
query_ncm_state = f"""
    SELECT e."SG_UF_NCM" AS uf,
           u.nome_estado AS state,
           e."CO_NCM" AS ncm_code,
           n.nome_ncm,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    JOIN ncm n ON e."CO_NCM" = n.codigo_ncm
    WHERE e."CO_ANO" = {MAX_YEAR}
    AND e."SG_UF_NCM" IN ({top10_states_uf_str})
    GROUP BY e."SG_UF_NCM", u.nome_estado, e."CO_NCM", n.nome_ncm
    ORDER BY e."SG_UF_NCM", exports_usd DESC
"""

df_ncm_state = pd.read_sql(query_ncm_state, engine)
df_ncm_state['exports_usd_bn'] = (df_ncm_state['exports_usd'] / 1e9).round(3)

df_top10_ncm_state = (
    df_ncm_state.groupby('state', group_keys=False)
    .apply(lambda x: x.nlargest(10, 'exports_usd'))
    .reset_index(drop=True)
)

print(f"Top 10 NCM products by state ({MAX_YEAR})")
for state in df_top10_ncm_state['state'].unique():
    subset = df_top10_ncm_state[df_top10_ncm_state['state'] == state]
    total  = df_state[df_state['uf'] == subset['uf'].iloc[0]]['exports_usd_bn'].iloc[0]
    print(f"\n{state} — Total exports: ${total:.1f}bn")
    for _, row in subset.iterrows():
        share = (row['exports_usd'] / (total * 1e9) * 100)
        print(f"  NCM {row['ncm_code']} {str(row['nome_ncm'])[:50]:<50} ${row['exports_usd_bn']:.3f}bn ({share:.1f}%)")

## ---- TOP NCM BY REGION ----
query_ncm_region = f"""
    SELECT u.nome_regiao AS region,
           e."CO_NCM" AS ncm_code,
           n.nome_ncm,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    JOIN ncm n ON e."CO_NCM" = n.codigo_ncm
    WHERE e."CO_ANO" = {MAX_YEAR}
    GROUP BY u.nome_regiao, e."CO_NCM", n.nome_ncm
    ORDER BY u.nome_regiao, exports_usd DESC
"""

df_ncm_region = pd.read_sql(query_ncm_region, engine)
df_ncm_region['exports_usd_bn'] = (df_ncm_region['exports_usd'] / 1e9).round(3)

df_top10_ncm_region = (
    df_ncm_region.groupby('region', group_keys=False)
    .apply(lambda x: x.nlargest(10, 'exports_usd'))
    .reset_index(drop=True)
)

## Regional totals
region_totals = df_region[df_region['year'] == MAX_YEAR].set_index('region')['exports_usd']

print(f"\nTop 10 NCM products by region ({MAX_YEAR})")
for region in df_top10_ncm_region['region'].unique():
    subset = df_top10_ncm_region[df_top10_ncm_region['region'] == region]
    total  = region_totals.get(region, 1)
    print(f"\n{region} — Total exports: ${total/1e9:.1f}bn")
    for _, row in subset.iterrows():
        share = (row['exports_usd'] / total * 100)
        print(f"  NCM {row['ncm_code']} {str(row['nome_ncm'])[:50]:<50} ${row['exports_usd_bn']:.3f}bn ({share:.1f}%)")

## 2.9 — State-Level China Dependency
Step 1 showed China absorbs 28.7% of Brazil's total exports. Is this uniform across all states or concentrated in specific ones?

In [ ]:
china_code_query = "SELECT codigo_pais FROM pais WHERE nome_pais_ing = 'China'"
china_code = pd.read_sql(china_code_query, engine)['codigo_pais'].iloc[0]

query_china_state = f"""
    SELECT e."SG_UF_NCM" AS uf,
           u.nome_estado AS state,
           SUM(e."VL_FOB") AS exports_to_china
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    WHERE e."CO_ANO" = {MAX_YEAR}
    AND e."CO_PAIS" = {china_code}
    GROUP BY e."SG_UF_NCM", u.nome_estado
    ORDER BY exports_to_china DESC
"""

df_china_state = pd.read_sql(query_china_state, engine)
df_china_state['exports_to_china_bn'] = (df_china_state['exports_to_china'] / 1e9).round(2)

df_china_share = df_china_state.merge(
    df_state[['uf', 'exports_usd', 'exports_usd_bn']], on='uf', how='left'
)
df_china_share['china_share_%'] = (
    df_china_share['exports_to_china'] / df_china_share['exports_usd'] * 100
).round(1)

df_china_plot = df_china_share.sort_values('china_share_%', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(df_china_plot['state'], df_china_plot['china_share_%'], color='steelblue', alpha=0.85)
ax.axvline(x=28.7, color='tomato', linestyle='--', linewidth=1.5, label='National average (28.7%)')
ax.set_title(f"Share of State Exports Destined for China ({MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_xlabel("China Share of State Exports (%)")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
plt.tight_layout()
plt.savefig('output_2_9_china_dependency.png', dpi=150)
plt.show()

print(f"\nChina dependency by state ({MAX_YEAR}) — national average: 28.7%")
print(df_china_share[['state', 'exports_to_china_bn', 'exports_usd_bn', 'china_share_%']]
      .sort_values('china_share_%', ascending=False)
      .to_string(index=False))

## 2.10 — Monthly Export Variation by State (Box Plot)
How much do monthly export values vary for each state? Uses the last 10 years (2016–2025) to capture recent structural patterns. A wide box indicates high seasonal or cyclical volatility; a narrow box indicates consistent monthly flows.

In [ ]:
query_monthly_exp = f"""
    SELECT e."CO_ANO" AS year,
           e."CO_MES" AS month,
           u.nome_estado AS state,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    WHERE e."CO_ANO" BETWEEN {BOX_START} AND {MAX_YEAR}
    AND e."SG_UF_NCM" IN ({top10_states_uf_str})
    GROUP BY e."CO_ANO", e."CO_MES", u.nome_estado
    ORDER BY u.nome_estado, year, month
"""

df_monthly_exp = pd.read_sql(query_monthly_exp, engine)
df_monthly_exp['exports_usd_bn'] = df_monthly_exp['exports_usd'] / 1e9

states_ordered = df_state.head(10)['state'].tolist()
monthly_data   = [df_monthly_exp[df_monthly_exp['state'] == s]['exports_usd_bn'].values
                  for s in states_ordered]

fig, ax = plt.subplots(figsize=(14, 7))
bp = ax.boxplot(monthly_data, labels=states_ordered, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='navy', linewidth=2))
ax.set_title(f"Monthly Export Variation by State ({BOX_START}–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_ylabel("Monthly Export Value (USD Billions)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fbn'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('output_2_10_monthly_exp_boxplot.png', dpi=150)
plt.show()

## 2.11 — Monthly Import Variation by State (Box Plot)
How much do monthly import values vary for each of the top 10 importing states?

In [ ]:
query_monthly_imp = f"""
    SELECT i."CO_ANO" AS year,
           i."CO_MES" AS month,
           u.nome_estado AS state,
           SUM(i."VL_FOB") AS imports_usd
    FROM imp i
    JOIN uf u ON i."SG_UF_NCM" = u.sigla
    WHERE i."CO_ANO" BETWEEN {BOX_START} AND {MAX_YEAR}
    AND i."SG_UF_NCM" IN ({top10_imp_states_uf_str})
    GROUP BY i."CO_ANO", i."CO_MES", u.nome_estado
    ORDER BY u.nome_estado, year, month
"""

df_monthly_imp = pd.read_sql(query_monthly_imp, engine)
df_monthly_imp['imports_usd_bn'] = df_monthly_imp['imports_usd'] / 1e9

imp_states_ordered = df_state.nlargest(10, 'imports_usd_bn')['state'].tolist()
monthly_imp_data   = [df_monthly_imp[df_monthly_imp['state'] == s]['imports_usd_bn'].values
                      for s in imp_states_ordered]

fig, ax = plt.subplots(figsize=(14, 7))
bp = ax.boxplot(monthly_imp_data, labels=imp_states_ordered, patch_artist=True,
                boxprops=dict(facecolor='tomato', alpha=0.6),
                medianprops=dict(color='darkred', linewidth=2))
ax.set_title(f"Monthly Import Variation by State ({BOX_START}–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_ylabel("Monthly Import Value (USD Billions)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fbn'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('output_2_11_monthly_imp_boxplot.png', dpi=150)
plt.show()

## 2.12 — State Trade Hubs
For each of the top 10 states, what is the export and import hub profile — dominant sectors, top trading partners and trade balance?

In [ ]:
## Top export partner per state
query_state_partner_exp = f"""
    SELECT e."SG_UF_NCM" AS uf,
           p.nome_pais_ing AS top_export_partner,
           SUM(e."VL_FOB") AS partner_exports
    FROM exp e
    JOIN pais p ON e."CO_PAIS" = p.codigo_pais
    WHERE e."CO_ANO" = {MAX_YEAR}
    AND e."SG_UF_NCM" IN ({top10_states_uf_str})
    GROUP BY e."SG_UF_NCM", p.nome_pais_ing
    ORDER BY e."SG_UF_NCM", partner_exports DESC
"""

## Top import partner per state
query_state_partner_imp = f"""
    SELECT i."SG_UF_NCM" AS uf,
           p.nome_pais_ing AS top_import_partner,
           SUM(i."VL_FOB") AS partner_imports
    FROM imp i
    JOIN pais p ON i."CO_PAIS" = p.codigo_pais
    WHERE i."CO_ANO" = {MAX_YEAR}
    AND i."SG_UF_NCM" IN ({top10_states_uf_str})
    GROUP BY i."SG_UF_NCM", p.nome_pais_ing
    ORDER BY i."SG_UF_NCM", partner_imports DESC
"""

df_sp_exp = pd.read_sql(query_state_partner_exp, engine)
df_sp_imp = pd.read_sql(query_state_partner_imp, engine)

## Keep only top partner per state
top_exp_partner = df_sp_exp.groupby('uf').first().reset_index()[['uf', 'top_export_partner']]
top_imp_partner = df_sp_imp.groupby('uf').first().reset_index()[['uf', 'top_import_partner']]

## Top export sector per state (from 2.6)
top_exp_sector = (
    df_exp_sectors.groupby('uf', group_keys=False)
    .apply(lambda x: x.nlargest(1, 'exports_usd'))
    .reset_index(drop=True)[['uf', 'descricao_sh2']]
    .rename(columns={'descricao_sh2': 'top_export_sector'})
)

## Top import sector per state (from 2.7)
top_imp_sector = (
    df_imp_sectors.groupby('uf', group_keys=False)
    .apply(lambda x: x.nlargest(1, 'imports_usd'))
    .reset_index(drop=True)[['uf', 'descricao_sh2']]
    .rename(columns={'descricao_sh2': 'top_import_sector'})
)

## Assemble hub profile
df_hub = df_state.head(10)[['uf', 'state', 'region', 'exports_usd_bn', 'imports_usd_bn', 'balance_usd_bn']]
df_hub = df_hub.merge(top_exp_partner, on='uf', how='left')
df_hub = df_hub.merge(top_imp_partner, on='uf', how='left')
df_hub = df_hub.merge(top_exp_sector,  on='uf', how='left')
df_hub = df_hub.merge(top_imp_sector,  on='uf', how='left')

print(f"State Trade Hub Profiles ({MAX_YEAR})")
print(df_hub.to_string(index=False))

## 2.13 — Regional Trade Hubs
Aggregating the state hub profiles to the regional level — what does each macro-region specialise in, and what are its key trade relationships?

In [ ]:
## Top export sector per region
query_region_exp_sector = f"""
    SELECT u.nome_regiao AS region,
           s.descricao_sh2 AS top_export_sector,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    JOIN ncm n ON e."CO_NCM" = n.codigo_ncm
    JOIN ncm_sh s ON n.codigo_sh6 = s.codigo_sh6
    WHERE e."CO_ANO" = {MAX_YEAR}
    GROUP BY u.nome_regiao, s.descricao_sh2
    ORDER BY u.nome_regiao, exports_usd DESC
"""

## Top import sector per region
query_region_imp_sector = f"""
    SELECT u.nome_regiao AS region,
           s.descricao_sh2 AS top_import_sector,
           SUM(i."VL_FOB") AS imports_usd
    FROM imp i
    JOIN uf u ON i."SG_UF_NCM" = u.sigla
    JOIN ncm n ON i."CO_NCM" = n.codigo_ncm
    JOIN ncm_sh s ON n.codigo_sh6 = s.codigo_sh6
    WHERE i."CO_ANO" = {MAX_YEAR}
    GROUP BY u.nome_regiao, s.descricao_sh2
    ORDER BY u.nome_regiao, imports_usd DESC
"""

## Top export partner per region
query_region_exp_partner = f"""
    SELECT u.nome_regiao AS region,
           p.nome_pais_ing AS top_export_partner,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    JOIN pais p ON e."CO_PAIS" = p.codigo_pais
    WHERE e."CO_ANO" = {MAX_YEAR}
    GROUP BY u.nome_regiao, p.nome_pais_ing
    ORDER BY u.nome_regiao, exports_usd DESC
"""

## Top import partner per region
query_region_imp_partner = f"""
    SELECT u.nome_regiao AS region,
           p.nome_pais_ing AS top_import_partner,
           SUM(i."VL_FOB") AS imports_usd
    FROM imp i
    JOIN uf u ON i."SG_UF_NCM" = u.sigla
    JOIN pais p ON i."CO_PAIS" = p.codigo_pais
    WHERE i."CO_ANO" = {MAX_YEAR}
    GROUP BY u.nome_regiao, p.nome_pais_ing
    ORDER BY u.nome_regiao, imports_usd DESC
"""

df_reg_exp_sec  = pd.read_sql(query_region_exp_sector,  engine)
df_reg_imp_sec  = pd.read_sql(query_region_imp_sector,  engine)
df_reg_exp_part = pd.read_sql(query_region_exp_partner, engine)
df_reg_imp_part = pd.read_sql(query_region_imp_partner, engine)

top_reg_exp_sec  = df_reg_exp_sec.groupby('region').first().reset_index()[['region', 'top_export_sector']]
top_reg_imp_sec  = df_reg_imp_sec.groupby('region').first().reset_index()[['region', 'top_import_sector']]
top_reg_exp_part = df_reg_exp_part.groupby('region').first().reset_index()[['region', 'top_export_partner']]
top_reg_imp_part = df_reg_imp_part.groupby('region').first().reset_index()[['region', 'top_import_partner']]

## Regional totals
df_region_totals = (
    df_state.groupby('region')
    .agg(exports_usd_bn=('exports_usd_bn', 'sum'),
         imports_usd_bn=('imports_usd_bn', 'sum'),
         balance_usd_bn=('balance_usd_bn', 'sum'))
    .reset_index()
)

df_region_hub = df_region_totals.copy()
df_region_hub = df_region_hub.merge(top_reg_exp_sec,  on='region', how='left')
df_region_hub = df_region_hub.merge(top_reg_imp_sec,  on='region', how='left')
df_region_hub = df_region_hub.merge(top_reg_exp_part, on='region', how='left')
df_region_hub = df_region_hub.merge(top_reg_imp_part, on='region', how='left')

print(f"Regional Trade Hub Profiles ({MAX_YEAR})")
print(df_region_hub.to_string(index=False))

## 2.14 — Commodities vs Processed Products by State and Region

> ⚠️ **PLACEHOLDER — Requires classification table**
>
> This cell requires a mapping table that classifies each NCM code (or SH2/SH4 code) as either a **raw commodity** or a **processed/manufactured product**. This classification does not exist in the current database and needs to be created before this analysis can be run.
>
> **Options for implementation (to be decided):**
> - Use `ncm_fat_agreg` categories and define a mapping of which `codigo_fat_agreg` values correspond to commodities vs processed goods
> - Use SH2 codes and manually classify each of the ~100 SH2 chapters as commodity or processed
> - Use `ncm_cgce_n1` classification which has a built-in economic use categorisation
> - Source an external commodity classification list (e.g. IMF primary commodity list) and map to NCM codes
>
> **What this cell will produce once implemented:**
> - % of exports classified as commodity vs processed for each state
> - % of exports classified as commodity vs processed for each region
> - Volume (USD bn) of commodity vs processed exports by state and region
> - Time series showing whether Brazil's export mix has shifted from commodities toward processed goods over 1997–2025

```python
## TO BE IMPLEMENTED — add commodity classification logic here
## Expected inputs: a DataFrame or SQL table mapping ncm_code -> 'commodity' or 'processed'
## Expected outputs: stacked bar charts and summary tables by state and region
```

## 2.15 — Key Findings

> *To be completed after running all cells above. Use the outline below as a guide.*

### Structure
1. **Which states dominate Brazilian trade** — top 3 exporting and importing states and their share of national totals
2. **Net exporters vs net importers** — how many states run surpluses vs deficits and any regional pattern
3. **State balance evolution** — which states were most affected by the 2011–2016 decline and which were resilient
4. **Regional specialisation** — what each macro-region specialises in based on hub profiles
5. **China dependency at state level** — which states are above and below the 28.7% national average, and what that implies
6. **Monthly volatility** — which states show the highest seasonal variation and what that means for market entry timing
7. **Limitations** — note any states with missing or low data, flag commodity vs processed analysis as pending
8. **What comes next** — Step 3 municipality-level hotspots